
# Tutorial: Cargar y Explorar un Dataset (Pandas)


**Objetivo**: Aprender a cargar archivos (CSV/Excel) en **Pandas**, explorar rápidamente su contenido (**Series** y **DataFrames**), y **combinar** (merge) distintos datasets usando `inner`, `left`, `right` y `outer`.



<a id="material"></a>
## 1) Material de estudio



<a id="series-df"></a>
### 1.1 Series vs DataFrames

- **Series**: estructuras **unidimensionales** con índice (etiquetas); análogas a una columna.  
- **DataFrames**: **tablas 2D** que agrupan múltiples Series por columnas.

> Idea clave: un `DataFrame` es una colección de **Series** bien alineadas por índice/filas. Esto permite filtrado, agregaciones y combinación flexibles.


In [1]:

import pandas as pd

print("Creando ejemplos de Series y DataFrame...")

serie = pd.Series([10, 20, 30], index=["a","b","c"])
print("Ejemplo de Series:")
print(serie)
print("Tipo de objeto:", type(serie))

df_demo = pd.DataFrame({
    "producto": ["A", "B", "C"],
    "precio": [1000, 1500, 1200],
    "stock": [10, 5, 20]
})
print("\nEjemplo de DataFrame:")
print(df_demo)
print("Tipo de objeto:", type(df_demo))

print("\nAcceder a una columna devuelve una Series:")
print("Tipo de df_demo['precio']:", type(df_demo["precio"]))

Creando ejemplos de Series y DataFrame...
Ejemplo de Series:
a    10
b    20
c    30
dtype: int64
Tipo de objeto: <class 'pandas.core.series.Series'>

Ejemplo de DataFrame:
  producto  precio  stock
0        A    1000     10
1        B    1500      5
2        C    1200     20
Tipo de objeto: <class 'pandas.core.frame.DataFrame'>

Acceder a una columna devuelve una Series:
Tipo de df_demo['precio']: <class 'pandas.core.series.Series'>



<a id="carga"></a>
### 1.2 Cargar archivos a DataFrames (CSV/Excel)

La carpeta **`datasets_tutorial3`** tiene los archivos necesarios. Este cuaderno intentará detectar la ubicación automáticamente.

**Archivos incluidos:**
- `clientes.csv`, `productos.csv`, `ventas.csv`
- `sucursales.xlsx`

Parámetros útiles en `read_csv` / `read_excel`:
- `dtype=` para fijar tipos (evita problemas en merges).
- `parse_dates=` para convertir columnas de fechas.
- `na_values=` / `keep_default_na=` para controlar valores faltantes.
- `encoding=` si hay caracteres especiales.


In [2]:
# Cargar CSV
clientes_path = "datasets_tutorial3/clientes.csv"
productos_path = "datasets_tutorial3/productos.csv"
ventas_path   = "datasets_tutorial3/ventas.csv"

print("\nLeyendo clientes, productos y ventas...")
clientes  = pd.read_csv(clientes_path, parse_dates=["fecha_registro"])
productos = pd.read_csv(productos_path)
ventas    = pd.read_csv(ventas_path, parse_dates=["fecha_venta"])

print("Clientes -> filas:", clientes.shape[0], "| columnas:", clientes.shape[1])
print("Productos -> filas:", productos.shape[0], "| columnas:", productos.shape[1])
print("Ventas -> filas:", ventas.shape[0], "| columnas:", ventas.shape[1])

#Cargar Excel
sucursales_xlsx = "datasets_tutorial3/sucursales.xlsx"

print("\nLeyendo sucursales desde Excel...")
sucursales = pd.read_excel(sucursales_xlsx, sheet_name="sucursales")
regiones   = pd.read_excel(sucursales_xlsx, sheet_name="regiones")


print("Sucursales ->", sucursales.shape, "| Regiones ->", regiones.shape)


Leyendo clientes, productos y ventas...
Clientes -> filas: 28 | columnas: 5
Productos -> filas: 12 | columnas: 4
Ventas -> filas: 120 | columnas: 6

Leyendo sucursales desde Excel...
Sucursales -> (5, 4) | Regiones -> (5, 2)



<a id="exploracion"></a>
### 1.3 Exploración rápida y chequeos iniciales

- Vista general: `head()`, `tail()`, `sample()`  
- Esquema: `dtypes`, `shape`, `columns`  
- Resumen numérico: `describe()`  
- Calidad de datos: `isna().sum()`, `duplicated().sum()`, `value_counts()`


In [3]:

print("Primeras filas de clientes:")
print(clientes.head())


Primeras filas de clientes:
   id_cliente     nombre                email        ciudad fecha_registro
0        1001    Joaquín       dup@htmail.com      Santiago     2024-11-23
1        1002  Valentina       dup@htmail.com         Talca     2024-09-08
2        1003  Alejandro  alejandro84@mail.cl  PUERTO MONTT     2024-03-05
3        1004     Emilia      emilia1@mail.cl       Iquique     2023-06-13
4        1005      Tomás      tomas98@mail.cl        Temuco     2023-04-05


In [4]:
print("\nValores faltantes por columna (clientes):")
print(clientes.isna().sum())


Valores faltantes por columna (clientes):
id_cliente        0
nombre            0
email             3
ciudad            0
fecha_registro    0
dtype: int64


In [5]:
print("\nCiudades más frecuentes (clientes['ciudad']):")
print(clientes["ciudad"].value_counts().head())



Ciudades más frecuentes (clientes['ciudad']):
Puerto Montt    5
Talca           3
Valparaíso      3
Concepción      3
Santiago        2
Name: ciudad, dtype: int64


In [6]:
print("\nResumen numérico de ventas.cantidad:")
print(ventas["cantidad"].describe())


Resumen numérico de ventas.cantidad:
count    120.000000
mean       3.066667
std        1.364208
min        1.000000
25%        2.000000
50%        3.000000
75%        4.000000
max        5.000000
Name: cantidad, dtype: float64



<a id="merge"></a>
### 1.4 Combinar datasets con `merge`

Casos principales:
- **INNER**: intersección de claves (solo las que matchean).
- **LEFT**: preserva todas las filas de la tabla de la **izquierda**.
- **RIGHT**: preserva todas las filas de la tabla de la **derecha**.
- **OUTER**: unión completa (todas las claves, matcheen o no).

Parámetros útiles:
- `on=` / `left_on=`, `right_on=` para especificar claves.
- `suffixes=("_x","_y")` para evitar choques de nombres.
- `indicator=True` para ver de dónde provienen las filas.


In [7]:

# 1) Enriquecer ventas con info de productos (precio)
print("INNER JOIN ventas x productos (solo claves coincidentes):")
v_prod_inner = pd.merge(ventas, productos, on="id_producto", how="inner")
print("Filas resultado (inner):", len(v_prod_inner))


INNER JOIN ventas x productos (solo claves coincidentes):
Filas resultado (inner): 112


In [8]:

# LEFT para conservar todas las ventas, incluso si falta el producto
print("\nLEFT JOIN ventas x productos (conservar todas las ventas):")
v_prod_left = pd.merge(ventas, productos, on="id_producto", how="left", indicator=True)
print(v_prod_left["_merge"].value_counts())




LEFT JOIN ventas x productos (conservar todas las ventas):
both          112
left_only       8
right_only      0
Name: _merge, dtype: int64


In [9]:
# Calcular total de la venta (cantidad * precio)
v_prod_left["total_venta"] = v_prod_left["cantidad"] * v_prod_left["precio"]
print("\nMuestra con total_venta calculado:")
print(v_prod_left[["id_venta","id_producto","cantidad","precio","total_venta"]].head())



Muestra con total_venta calculado:
   id_venta  id_producto  cantidad    precio  total_venta
0     50001         2005         2  159990.0     319980.0
1     50002         2006         4  219990.0     879960.0
2     50003         2008         4   99990.0     399960.0
3     50004         2006         4  219990.0     879960.0
4     50005         2012         3   14990.0      44970.0


In [10]:
# 2) Unir con clientes (LEFT) para enriquecer con ciudad/registro
print("\nLEFT JOIN ventas x clientes:")
v_cli = pd.merge(ventas, clientes, on="id_cliente", how="left", indicator=True, validate="many_to_one")
print(v_cli["_merge"].value_counts())


LEFT JOIN ventas x clientes:
both          115
left_only       5
right_only      0
Name: _merge, dtype: int64


In [11]:
# 3) Detectar claves no matcheadas con OUTER e indicator
print("\nOUTER JOIN ventas x clientes para detectar faltantes en ambos lados:")
outer = pd.merge(ventas, clientes, on="id_cliente", how="outer", indicator=True)
print(outer["_merge"].value_counts())



OUTER JOIN ventas x clientes para detectar faltantes en ambos lados:
both          115
left_only       5
right_only      0
Name: _merge, dtype: int64


In [12]:
# 4) Agregar región de sucursales y luego el país (merge encadenado)
print("\nLEFT JOIN ventas x sucursales -> agregar región; luego unir regiones para agregar país:")
v_suc = pd.merge(ventas, sucursales, on="sucursal_id", how="left")
v_suc_reg = pd.merge(v_suc, regiones, on="region", how="left")
print("Columnas finales:", list(v_suc_reg.columns))


LEFT JOIN ventas x sucursales -> agregar región; luego unir regiones para agregar país:
Columnas finales: ['id_venta', 'id_cliente', 'id_producto', 'fecha_venta', 'cantidad', 'sucursal_id', 'sucursal', 'ciudad', 'region', 'pais']



<a id="buenas-practicas"></a>
### 1.5 Errores comunes y buenas prácticas

- **Tipos**: asegúrate de que las claves tengan el **mismo dtype** en ambos DataFrames.  
- **Duplicados**: verifica `duplicated(subset=[clave])` cuando esperas relaciones `many_to_one` o `one_to_one`.  
- **Valores faltantes** tras `merge`: úsalos como señal para *diagnosticar* claves que no matchearon.  
- **`indicator=True`** ayuda a auditar rápidamente de dónde vienen las filas.  






<a id="ejercicios"></a>
## 2) Ejercicios

> **Formato**: usa impresiones descriptivas en cada paso para que el output sea claro (por ejemplo: `print("Filas en clientes:", len(clientes))`).




### Ejercicio 1 — Carga básica
1. Lee `clientes.csv`, `productos.csv`, `ventas.csv` y `sucursales.xlsx` (o los CSV equivalentes).  
2. Imprime **filas y columnas** de cada DataFrame y verifica `dtypes`.



### Ejercicio 2 — Exploración rápida
1. Muestra `head()` de cada DataFrame.  
2. En `clientes`, reporta cuántos valores faltantes hay por columna e identifica **emails duplicados**.  
3. En `ventas`, calcula `describe()` de `cantidad` y el **número de fechas únicas**.



### Ejercicio 3 — Enriquecer ventas con productos
1. Haz un **LEFT join** `ventas × productos` por `id_producto`.  
2. Crea `total_venta = cantidad * precio`.  
3. Identifica las **3 categorías** con mayor **recaudación total** (suma de `total_venta`).



### Ejercicio 4 — Ventas con y sin cliente
1. Haz un **LEFT join** `ventas × clientes`.  
2. ¿Cuántas ventas **no** encontraron cliente? Lista los `id_cliente` problemáticos.  
3. ¿Cuántos clientes **no** tienen ventas? (pista: `outer` + `indicator`).



### Ejercicio 5 — Sucursales y regiones
1. Une `ventas` con `sucursales` y luego con `regiones`.  
2. Calcula la **recaudación total por región** y ordena de mayor a menor.  
3. (Opcional) Guarda el resultado como `recaudacion_por_region.csv`. (usa .to_csv()) si quisieras guardar .xlsx usa to_excel()